# Tool Calling (Function Calling)

함수 선언, Multi-turn 함수 결과 전달, Built-in Tools (Google Search, Code Execution).

In [1]:
from llm_request._internal import handler_client as llm_handler
from llm_request._internal.handler_client import call

# llm_handler.configure(base_url="http://localhost:8100")


### 5-1. Function Declaration — 함수 선언 규칙

각 함수는 **dict 하나**로 정의하며, `tools=` 리스트에 넣어 전달한다.

```python
{
    "name": str,           # (필수) 함수 이름 — 영문, 언더스코어 사용
    "description": str,    # (필수) 함수가 하는 일 — 모델이 이걸 보고 호출 여부를 판단
    "parameters": {        # (필수) JSON Schema 형식의 파라미터 정의
        "type": "object",
        "properties": {
            "param_name": {
                "type": "string",       # string | integer | number | boolean | array
                "description": "설명",  # 모델이 어떤 값을 넣을지 판단하는 근거
                "enum": ["A", "B"],     # (선택) 허용 값 제한
            },
        },
        "required": ["param_name"],  # (선택) 필수 파라미터 목록
    },
}
```

**핵심 규칙:**
- `description`이 **가장 중요** — 모델은 이 설명을 보고 어떤 함수를 호출할지 결정한다
- `parameters.properties`의 각 파라미터에도 `description`을 달아야 모델이 정확한 값을 생성
- `parameters.type`은 항상 `"object"`
- 여러 함수를 정의하면 모델이 상황에 맞는 함수를 자동 선택

In [9]:
# 여러 함수를 tools 리스트에 정의 → 모델이 상황에 맞는 함수를 자동 선택
tools = [
    {
        "name": "get_stock_price",  # 함수 이름
        "description": "주식의 현재 가격을 조회합니다",  # 모델이 이걸 보고 호출 여부 판단
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "종목코드 (예: 005930)",  # 모델이 어떤 값을 넣을지 근거
                },
            },
            "required": ["ticker"],
        },
    },
    {
        "name": "get_exchange_rate",
        "description": "환율을 조회합니다",
        "parameters": {
            "type": "object",
            "properties": {
                "currency": {
                    "type": "string",
                    "description": "통화코드 (예: USD, JPY)",
                    "enum": ["USD", "JPY", "EUR", "CNY"],  # 허용 값 제한
                },
            },
            "required": ["currency"],
        },
    },
]

# 모델이 "삼성전자 주가" → get_stock_price 자동 선택
response = call("삼성전자 주가 알려줘", tools=tools)

# function call 응답: text는 None, function_calls에 호출 정보
print("text:", response.text)  # None
print(
    "function_calls:", response.function_calls
)  # [FunctionCall(name="get_stock_price", ...)]

text: None
function_calls: [FunctionCall(name='get_stock_price', args={'ticker': '005930'}, id='0uktmaw8', thought_signature='Er0CCroCAQw51scbwGgyuUX3eSlPIEEyykZLpTI4115hkTifeTgircVOcH94f/nyeTS7BNz3+kT4tgfzBsbksukXcH22w9ZIdvGb76jjQd20+hryg+DR6NK2T9RLkOJOwe8m1YCEavszo+eJybnQWsnD47J+x/B1XDJe8t6zBOKfyYGerOOOynqwu/FA1YQkZ4baejS8e4TUG//VYfeHwI9vCJVBKoTy9CRIJvGvr861QOCxU4XiqcnmvmExPbmbfI4zmBF6blcH+1Z0MFGDthUlv22eQhGL2PLqPx+EYoD+fReSZoza/nnKkKzpeeVL2Kk03TrdPJDcOwAY9m+DUrA46eYAWFsz9V0lv8LFn7OIy4fRYLPfPe+Fee4iQuj9iy+4BioAfZ6fFntUxsp8bxbeivzkjL/5M8befe5DEf4=')]


### 5-2. Function Call 응답 구조 확인

`response.function_calls`는 `list[FunctionCall]` — 각 항목에 `name`, `args`, `id`, `thought_signature` (Gemini 3).

Gemini 3 부터 **thought_signature** 가 함수 호출과 함께 옴 — round-trip 시 history 에 보존 필수.


In [ ]:
if response.function_calls:
    for fc in response.function_calls:
        print(f"함수: {fc.name}")
        print(f"인자: {fc.args}")
        print(f"ID:   {fc.id}")
        ts = fc.thought_signature
        print(
            f"ts:   {ts[:30] + '...' if ts else None} ({len(ts) if ts else 0} chars base64)"
        )


함수: get_stock_price
인자: {'ticker': '005930'}
ID:   akvo2szq
ts:   ErwDCrkDAQw51sfw//repQLccRR6sW... (596 chars base64)


### 5-3. Multi-turn — `Session.send_tool_results()`

에이전트가 함수를 실행한 뒤 결과를 회신하면 Gemini 가 최종 응답을 만든다.

**`Session` 을 쓰면 history / role / `assistant_message` echo / `thought_signature` 보존을 모두 자동 처리**.
사용자는 함수 실행 결과만 `(id, name, response_dict)` tuple 로 회신하면 됨.

⚠️ **`tools=` 는 Turn 2 에도 동일하게 전달 필요** — Gemini 가 function 정의를 잃으면 `function_response` 를 해석하지 못해 빈응답 발생.
SDK 의 `send_tool_results` 가 누락 시 ValueError 로 fail-fast.


In [4]:
session = llm_handler.session()

# Turn 1 — function call 요청 받기
resp1 = session.call("삼성전자(005930) 주가 알려줘", tools=tools)
fc = resp1.function_calls[0]
print(f"Turn 1: 모델이 {fc.name}({fc.args}) 호출 요청, id={fc.id}")

# 에이전트가 함수 실행 (여기서는 더미)
function_result = {"price": 87500, "currency": "KRW", "change": "+1.2%"}

# Turn 2 — 함수 결과 회신 → 최종 텍스트.
# tools= 는 Turn 1 과 동일하게 다시 전달 (Gemini 가 function 정의 잃지 않도록).
# 누락 시 SDK 안전망이 ValueError 로 차단.
resp2 = session.send_tool_results([(fc.id, fc.name, function_result)], tools=tools)
print(f"Turn 2: {resp2.text}")


Turn 1: 모델이 get_stock_price({'ticker': '005930'}) 호출 요청, id=4gvp0z6h
Turn 2: 삼성전자의 현재 주가는 **87,500원**입니다.

이전 거래일 대비 **1.2% 상승**한 상태입니다. (실시간 정보는 시장 상황에 따라 지연될 수 있습니다.)


### 5-3b. 수동 round-trip — `call(messages=...)` 명시 path (advanced)

Session 없이 `call()` 만으로 history 를 직접 만들고 싶다면 `messages=[Content, ...]` 명시.
- `Content` / `ContentPart` Pydantic 모델로 type-safe 구성 — role 실수 / 오타 / 잘못된 part type 이 정적 분석으로 잡힘.
- `response.assistant_message` 는 `Content` 객체 그대로 전달 가능 — `.model_dump()` 변환 불필요.
- **변형/필터링 시 thought_signature bytes 손상 → 400 INVALID_ARGUMENT**.


In [ ]:
from llm_request._internal.handler_client import call, Content, ContentPart, FunctionResponse

# Turn 1
r1 = call("삼성전자(005930) 주가 알려줘", tools=tools)
fc = r1.function_calls[0]

# Turn 2 — call(messages=...) 명시 path. role 실수 차단 + type-safe.
r2 = call(
    messages=[
        Content(role="user", content="삼성전자(005930) 주가 알려줘"),
        r1.assistant_message,  # Content 객체 그대로 (role="model" + parts)
        Content(
            role="tool",
            content=[
                ContentPart(
                    type="function_response",
                    function_response=FunctionResponse(
                        name=fc.name,
                        response={"price": 87500, "currency": "KRW"},
                        id=fc.id,
                    ),
                ),
            ],
        ),
    ],
    tools=tools,
)
print(r2.text)


삼성전자(005930)의 현재 주가는 87,500원입니다.


### 5-4. tool_config — Function Calling 모드

| 모드 | 동작 |
| --- | --- |
| `"auto"` | 모델이 판단 (기본값) |
| `"any"` | 반드시 함수 호출 |
| `"none"` | 함수 호출 금지 (텍스트만) |

In [6]:
# 반드시 함수를 호출하도록 강제
response = call(
    "삼성전자에 대해 알려줘",
    tools=tools,
    tool_config="any",
)
print("function_calls:", response.function_calls)  # 반드시 function call 반환

function_calls: [FunctionCall(name='get_stock_price', args={'ticker': '005930'}, id='5e71fg8p', thought_signature='EvkCCvYCAQw51seF24cF0Im0su2ZtyplyjMMiiTJtnQWourQ1cGiBVhsIe6slNsk5Ug8GBTKM6o3IobkvqPJLQz7YsfVgEk49tWzxgzJ0TjPgaOH7DZTVFiQTSialXPj2DKP49nuFvkggvOeBtvNLXyNaAkXmGZri0q2wutTKqUNofcfekd12Y6eCRTLEp87JHsv9K/ljsOTKSMlMdWkedlVgavlojSoYUF713E/3zl49hIO54QsBqJvMqUneCrGpIYR8oambjcCk/COaefOyKUvMvDQx4K9OStv74RntoUcy865nN7Ycl5CcBP2zBJJlhUaukEpE6oxFL/53iSdzv0l1ReTrJtGMdYxk6TKR5tCtBhZ30u8j+zomAbbYvP4acZCEDLLQLJCHO1gZrYR7hG59Iic8gdAu9xsobdsBAALpOv5dhxSppZp/8ACP2RhAFXen7J+biyrZxbnJoWwK/IW612WbsTbpF5dQryp+LQS+usWP0rGfyJBNeI=')]


### 5-5. Function Declaration + Output Schema

Tool calling과 구조화된 출력을 **동시에** 사용할 수 있다 (Gemini 3 모델).

**두 가지 시나리오:**

| 시나리오 | 모델 동작 | 응답 위치 |
| --- | --- | --- |
| 모델이 함수 호출 결정 | `function_calls`에 호출 정보 | `response.function_calls` |
| 모델이 직접 답변 결정 | `schema`에 맞는 JSON 텍스트 | `response.text` |

→ 모델이 함수를 부르든 직접 답하든, **출력 형식이 예측 가능**.

In [7]:
# tools + output schema 동시 사용
# → 모델이 함수를 안 부르고 직접 답할 때, JSON schema를 따르도록 강제
response = call(
    "삼성전자 현재 주가가 얼마야?",
    tools=tools,
    output="json",
    schema={
        "type": "object",
        "properties": {
            "answer": {"type": "string"},
            "source": {"type": "string", "enum": ["function_call", "direct"]},
        },
    },
)

if response.function_calls:
    print("모델이 함수 호출을 선택:", response.function_calls)
else:
    print("모델이 직접 답변 (JSON schema 준수):", response.text)

모델이 함수 호출을 선택: [FunctionCall(name='get_stock_price', args={'ticker': '005930'}, id='he854hw4', thought_signature='EqUDCqIDAQw51scpkfoGz88H2zGf3OIJ32FQHdrDASPW5p0wzRPVz17j/2OUyK5cJEDsWbCWH4RPutNZ0HUYR57vI5ZDL8huUbbSAQHVFkODNltUJuq9cgZlj5QTaY1x7mKDuN9Hw66xQGlqndz2Muz5p8r9vUNB5lxFjp1Dj7iDluVTDZzVqeCBGbckiqj17aYNpvpwLJyu78NPNr1y61lIitBSJH2M3qM54iNWyyo6QsUVFDeTZGnMbTERrlJJeMBB1CRLNG2oFuiNOYxNhRCAdiUHXKyfFk7u1rNvEGgdnYQ1V2jKNjLXxL6+uV7/dToqhMcNtnjEjvCU+f85aqG+/DPS02ZhkxYLPiJdWWIfcl4o430Svc2bgKRnYM7Y7260StOZm59RfA8HWRYWau5gF7pgNW+m9tcpXFXgBxfHqwiIjSXULskbbjm0l0QWNjfHIyN4z+vJm5jf/vmtKkcqM7a18ERCNZThzfp8+nEdmD8yQxP3uesxTYNie0OGHJrcH3ikZkbvjm0spC6BzHSXJybhdz8zffy/hO2xc8wXsiMAvJjWHQ==')]


#### `response_json_schema` — 함수 반환값 구조 정의

FunctionDeclaration에 `response_json_schema` 필드를 추가하면,
모델에게 "이 함수의 반환값은 이런 구조"라고 알려줄 수 있다.
Multi-turn에서 함수 결과를 해석할 때 더 정확해진다.

In [8]:
# 함수 선언에 response_json_schema 추가
tools_with_response_schema = [
    {
        "name": "get_stock_price",
        "description": "주식의 현재 가격을 조회합니다",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string", "description": "종목코드"},
            },
            "required": ["ticker"],
        },
        "response_json_schema": {  # 함수 반환값 구조를 모델에게 알려줌
            "type": "object",
            "properties": {
                "price": {"type": "number"},
                "currency": {"type": "string"},
                "change_percent": {"type": "number"},
            },
        },
    },
]

response = call("삼성전자 주가", tools=tools_with_response_schema)
print(response.function_calls)

[FunctionCall(name='get_stock_price', args={'ticker': '005930.KS'}, id='9oknqu8v', thought_signature='EqMECqAEAQw51sfJwZY6EUkK5n8DKqndvUFQhYman465smWZ91n3wkR/kTtcfo9jAYNphbUJdGVftHcvym56jcjCUCFrdCz6xktGLGkOLOdTFzIzcqhX0pd8G0Chri/68OZIGXZDO3eMWwvWbiMuiXO+vOTLuy9RFjkQYhWO80jP+4hJ+sV52leKip5m2XVuVo/IDscCfRo1XTZYmO+wulhpZl3CASvozDJJ3GqdKE9rPpzchuGZynLPfA8h9jT8rM054DSIDumXwa/tEoQ48pzB4kLbCkHgIMY8rOU8Y8YqGXnTuNeW6bGnFdY+16bm0kAXzUEoomXiIKBvCS/541O5F/TIVnCWwGn6S7zSD53xoumb3Z8olDHvuWoaILoD42/yr4N7VXP/W288bsgxlO4qhx/8tGyXewOC5NU8Tr7VEnmdw/dJ+4X3DIfRC4FCFFQQnXyCcwOynXpOWU/vn1dVPbfIv8SYqqGZvC53s48DKSvJRUh+yLV7atYVeGCGmAjY/Qkg7BuLGK4QFnqZjA+vvdGjAgo2VKw+k8vUbamnR1NVkzfns4EPtEzpPJ7Q3mdEl4jeONItqNeXldYwNZg5squRq5tHaEvLmA8MO0a6iE48MSkjlzqiRrvhV01mF99m/zWTFXcZ7x098WYQxwbA9drH24WlvfRWbe0I3ugqtG19YV3zjhipln1JeWkO4M20X3e+HMGgYBwRO5SYwJ8i2HO41A==')]


### 5-6. Built-in Tools — Google Search + Grounding

google_search 결과는 **`response.grounding`** 에 정리되어 옴:

- `web_search_queries`: Gemini 가 실제로 검색한 쿼리들
- `chunks`: 인용한 출처 (title + uri). **uri 는 vertexai redirect, 수시간~1일 ephemeral**
- `supports`: 답변 text 의 어느 span 이 어느 chunk 를 참조했는지


In [ ]:
response = call(
    "오늘 코스피 지수 알려줘",
    tools=[{"google_search": {}}],
)
print("text:", response.text)
print()

if response.grounding:
    print("검색 쿼리:", response.grounding.web_search_queries)
    print("출처:")
    for i, c in enumerate(response.grounding.chunks):
        print(f"  [{i}] {c.title}: {c.uri[:80]}...")
    print("인용 매핑:")
    for s in response.grounding.supports[:3]:
        if s.segment:
            txt = response.text[s.segment.start_index : s.segment.end_index]
            print(f"  chunks={s.chunk_indices}: '{txt}'")


text: 2026년 5월 11일(월) 코스피 지수는 전 거래일 대비 **324.24포인트(+4.32%) 급등한 7,822.24**로 장을 마감했습니다.

오늘 시장의 주요 특징은 다음과 같습니다.

*   **사상 최고치 경신:** 코스피 지수가 역사상 처음으로 7,800선을 돌파하며 5거래일 연속 사상 최고 기록을 갈아치웠습니다. 장 중 한때는 7,899.32까지 치솟으며 7,900선 문턱까지 가기도 했습니다.
*   **반도체 주의 독주:** 인공지능(AI) 반도체 수요 기대감과 수출 호조에 힘입어 **삼성전자(+6.42%, 285,750원)**와 **SK하이닉스(+12.99%, 1,905,000원)** 등 대형 반도체 종목이 지수 상승을 강력하게 견인했습니다.
*   **매수 사이드카 발동:** 오전 중 지수가 급격히 상승하면서 프로그램 매수 호가 효력을 일시 정지하는 '매수 사이드카'가 발동되기도 했습니다.
*   **시장 양극화:** 지수는 크게 올랐지만, 삼성전자와 SK하이닉스를 제외한 상당수 종목은 오히려 하락하는 '기형적 장세'가 나타났습니다. 실제로 유가증권시장에서 상승 종목(183개)보다 하락 종목(693개)이 훨씬 많았습니다.
*   **코스닥 지수:** 코스피의 폭등과 달리 코스닥은 전 거래일보다 0.38포인트(-0.03%) 소폭 하락한 **1,207.34**로 마감하며 온도 차를 보였습니다.

오늘 시장은 반도체 업황에 대한 강력한 낙관론이 지배했으나, 지수 상승의 온기가 시장 전체로 퍼지지 못한 점이 특징적이었습니다.

검색 쿼리: ['KOSPI index May 11 2026', '오늘 코스피 지수 현황 2026년 5월 11일', '2026년 5월 11일 코스피 지수']
출처:
  [0] hankyung.com: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEYQSSPh9g7h...
  [1] koreatimes.co.kr: https://vertexai

### 5-7. Built-in Tools — Code Execution

code_execution 결과는 **`response.code_blocks`** (코드) + **`response.code_results`** (실행 결과) 로 분리.

- `CodeBlock.language` — 보통 `"PYTHON"`
- `CodeResult.outcome` — `"OUTCOME_OK"` / `"OUTCOME_FAILED"` / `"OUTCOME_DEADLINE_EXCEEDED"`


In [10]:
response = call(
    "피보나치 수열의 20번째 항을 계산해줘",
    tools=[{"code_execution": {}}],
)
print("text:", response.text)
print()

if response.code_blocks:
    for i, blk in enumerate(response.code_blocks):
        print(f"[code {i}, lang={blk.language}]")
        print(blk.code)
        print()
if response.code_results:
    for i, r in enumerate(response.code_results):
        print(f"[result {i}, outcome={r.outcome}]")
        print(r.output)


text: 피보나치 수열의 20번째 항($F_{20}$)은 **6,765**입니다.

수열의 처음 몇 항은 다음과 같습니다:
1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, **6765**, ...

[code 0, lang=PYTHON]
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

print(f"F_20 = {fibonacci(20)}")


[result 0, outcome=OUTCOME_OK]
F_20 = 6765



### 5-8. FunctionDeclaration + Built-in Tool 혼합 (Gemini 3 mixed paradigm)

두 종류 도구를 같이 쓰면 Gemini 3 가 `tool_call` / `tool_response` part 를 응답에 추가.
handler 는 `tools=` 에 둘 다 보이면 **`include_server_side_tool_invocations=True` 자동 활성**.

응답에 노출되는 신규 필드:
- `response.tool_calls`: server-side tool 호출 흔적 (`tool_type=GOOGLE_SEARCH_WEB` 등)
- `response.tool_responses`: 위 호출 결과 (id 로 짝)
- `response.assistant_message`: role="model" + 모든 part 통째 (function_call + tool_call + tool_response) — round-trip echo 용


In [7]:
mixed_tools = [
    {
        "name": "get_stock_price",
        "description": "주식의 현재 가격을 조회합니다",
        "parameters": {
            "type": "object",
            "properties": {"ticker": {"type": "string"}},
            "required": ["ticker"],
        },
    },
    {"google_search": {}},
]

session = llm_handler.session()

# Turn 1 — 모델이 함수 호출 + 백그라운드 검색 모두 시도
r1 = session.call(
    "삼성전자(005930) 현재 주가를 get_stock_price 함수로 조회하고, "
    "최근 분기 영업이익을 검색해서 비율을 계산해줘.",
    tools=mixed_tools,
)
print("Turn 1 assistant_message.content part types:")
if r1.assistant_message:
    for p in r1.assistant_message.content:
        print(f"  - {p.type}")

if r1.tool_calls:
    for tc in r1.tool_calls:
        print(f"server tool 호출: {tc.tool_type}, args={tc.args}")

fc = r1.function_calls[0] if r1.function_calls else None
if fc:
    print(f"client function 호출: {fc.name}({fc.args})")

    # Turn 2 — 함수 결과 회신 → 최종 답변 + grounding
    r2 = session.send_tool_results(
        [(fc.id, fc.name, {"price": 87500, "currency": "KRW"})],
        tools=mixed_tools,
    )
    print("\nTurn 2 최종 답변:")
    print(r2.text)
    if r2.grounding:
        print("\n출처:")
        for c in r2.grounding.chunks[:5]:
            print(f"  - {c.title}")


Turn 1 assistant_message.content part types:
  - function_call
  - tool_call
  - tool_response
server tool 호출: GOOGLE_SEARCH_WEB, args={'queries': ['삼성전자 2026년 1분기 영업이익', '삼성전자 가장 최근 분기 영업이익']}
client function 호출: get_stock_price({'ticker': '005930'})

Turn 2 최종 답변:
삼성전자(005930)의 현재 주가와 최근 분기 영업이익을 바탕으로 계산한 결과는 다음과 같습니다.

### 1. 주요 데이터 조회
*   **현재 주가:** 87,500원 (조회 시점 기준)
*   **최근 분기(2026년 1분기) 영업이익:** 약 57조 2,328억 원 (잠정치 기준 역대 최대 실적)
*   **상장 주식 수:** 약 5,846,278,608주 (보통주 기준)

### 2. 비율 계산
주가와 영업이익을 직접 비교하기 위해 **'시가총액 대비 분기 영업이익 비율'** 또는 **'주가 대비 주당 영업이익 비율'**을 계산할 수 있습니다.

*   **시가총액:** 주가(87,500원) × 주식 수(약 58.46억 주) ≒ **약 511조 5,494억 원**
*   **주당 영업이익(Quarterly OPPS):** 분기 영업이익(57.23조 원) ÷ 주식 수(58.46억 주) ≒ **약 9,789원**

#### 계산 결과 (시가총액 / 분기 영업이익)
> **약 8.94배**

### 3. 분석 요약
계산된 비율 **8.94**는 현재의 시가총액이 최근 한 분기 동안 벌어들인 영업이익의 약 9배 수준임을 의미합니다. 이를 연간으로 환산(분기 이익 × 4)할 경우, 영업이익 대비 시가총액 배수(EV/EBIT와 유사한 개념)는 **약 2.24배** 수준으로 떨어집니다.

이는 2026년 1분기 삼성전자가 반도체 슈퍼사이클과 AI 수요 폭증으로 인해 기록적인 영업이익(57조 원

### 5-9. `tool_config="validated"` — AUTO + 화이트리스트 (Gemini 3)

`"validated"` 는 AUTO 의 자율성 (함수 호출/직접 답변 선택) + `allowed_function_names` 화이트리스트 강제.
민감 함수가 있는 시나리오에서 `"any"` (강제 호출) 보다 안전.


In [10]:
response = call(
    "삼성전자 주가 알려줘",
    tools=tools,
    tool_config="validated",  # AUTO + 화이트리스트
)
print("function_calls:", response.function_calls)
print("text:", response.text)


function_calls: [FunctionCall(name='get_stock_price', args={'ticker': '005930'}, id='9venpdzu', thought_signature='EpIDCo8DAQw51sc2ey56AIMXaSlR0Pewn5lwCCAcKzQtqb7RqKEXD22+BhZSvP5R9D8aWgXHiTZl9IgOYYuHhumrWL0Gm0oQbm915l3jzRwNgMTN1+vQC0C5wgLxUqGNaDlk/RxjI3R72p3IHabHAPyNP6dparZWcQ+sHQHbalfyoAtdUEM/uL7k497iCTAHRMi/R1lV5dpyHygZbkTj0s8wmuqSJxlW9NuHAI5rPEHLGpEAY/lgCItGaOzsAEcuhJY6KO1k92KXC1Afi1cVN2dmVoZp5hwOkt4yrHNTj7IY0uB4bJG2BrQGEgTXn7sKIijB3L8X8sLqvpcBzUoEWVSxutrqmI+FposYmtp1nZ6X6izsqTWeTb9dU14faVpGnG/9MJ7wWVLfkfa+9skaEl2mgZXs4bkUs2egPQcm6ePTKPEAojPqqiSRNYx0yVQepPZtkSTuvRnyBObbiSgwb6KcY9mLUss1fR6OfCSs6YgfOCj7WCR1HCv7z96Mk33mY8OfXB5p2q8WsoElHeMGcOXTjsbR')]
text: None


### 5-10. error_type으로 에러 출처 확인

| error_type | 의미 | 대응 |
| --- | --- | --- |
| `validation` | 요청 오류 | 파라미터 수정 |
| `provider` | Gemini API 실패 | 재시도 가능 |
| `handler` | Handler 내부 버그 | Handler 팀 보고 |


In [ ]:
from llm_request._internal.handler_client.exceptions import HandlerError

try:
    response = call("테스트", tools=[{"name": "test"}])  # parameters 누락
except HandlerError as e:
    print(f"error_type: {e.error_type}")
    print(f"status:     {e.status_code}")
    print(f"message:    {e.message}")